# Shared VOC Localisation Starter

This notebook enforces the same train/val/test IDs for everyone.

Pipeline:
1. Load locked splits from artifacts/group_split when available.
2. Build train/val/test object tables.
3. Train model-specific detector.
4. Export predictions to artifacts/predictions/*.csv using a common schema.
5. Evaluate mAP@0.5 on validation split with shared code.

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

In [ ]:
PROJECT_ROOT = Path.cwd()
TRAINVAL_ROOT = PROJECT_ROOT / 'trainval' / 'VOC2012'
TEST_ROOT = PROJECT_ROOT / 'test' / 'VOC2012'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
SPLIT_DIR = ARTIFACTS_DIR / 'group_split'
PREDICTIONS_DIR = ARTIFACTS_DIR / 'predictions'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
CLASS_TO_ID = {name: idx for idx, name in enumerate(VOC_CLASSES)}
ID_TO_CLASS = {idx: name for name, idx in CLASS_TO_ID.items()}

PREDICTION_COLUMNS = [
    'image_id', 'split', 'source_model', 'class_id', 'cls',
    'xmin', 'ymin', 'xmax', 'ymax', 'score'
]

for path in [TRAINVAL_ROOT, TEST_ROOT]:
    if not path.exists():
        raise FileNotFoundError(f'Missing expected VOC folder: {path}')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SPLIT_DIR:', SPLIT_DIR)

In [ ]:
def read_ids_from_file(path: Path) -> list[str]:
    if not path.exists():
        return []
    return [line.strip() for line in path.read_text().splitlines() if line.strip()]


def load_split_ids(split_name: str) -> list[str]:
    # Prefer locked team split files to avoid leakage.
    shared_path = SPLIT_DIR / f'{split_name}_ids.txt'
    shared_ids = read_ids_from_file(shared_path)
    if shared_ids:
        return shared_ids

    # Fallback to VOC default lists if shared split files are missing.
    root = TRAINVAL_ROOT if split_name in {'train', 'val'} else TEST_ROOT
    fallback_path = root / 'ImageSets' / 'Main' / f'{split_name}.txt'
    return read_ids_from_file(fallback_path)


def parse_annotation(xml_path: Path, image_id: str, split_name: str) -> tuple[dict, list[dict]]:
    root = ET.parse(xml_path).getroot()

    size_node = root.find('size')
    width = int(size_node.findtext('width', default='0')) if size_node is not None else 0
    height = int(size_node.findtext('height', default='0')) if size_node is not None else 0

    records = []
    for obj in root.findall('object'):
        cls = obj.findtext('name', default='').strip()
        if cls not in CLASS_TO_ID:
            continue

        box = obj.find('bndbox')
        if box is None:
            continue

        xmin = int(float(box.findtext('xmin', default='0')))
        ymin = int(float(box.findtext('ymin', default='0')))
        xmax = int(float(box.findtext('xmax', default='0')))
        ymax = int(float(box.findtext('ymax', default='0')))

        box_w = max(1, xmax - xmin + 1)
        box_h = max(1, ymax - ymin + 1)

        records.append({
            'image_id': image_id,
            'split': split_name,
            'cls': cls,
            'class_id': CLASS_TO_ID[cls],
            'xmin': xmin,
            'ymin': ymin,
            'xmax': xmax,
            'ymax': ymax,
            'box_w': box_w,
            'box_h': box_h,
            'box_area': box_w * box_h,
            'difficult': int(obj.findtext('difficult', default='0')),
            'truncated': int(obj.findtext('truncated', default='0')),
            'occluded': int(obj.findtext('occluded', default='0')),
            'width': width,
            'height': height,
        })

    image_meta = {
        'image_id': image_id,
        'split': split_name,
        'width': width,
        'height': height,
    }
    return image_meta, records


def build_split_tables(root_dir: Path, split_name: str, image_ids: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    image_rows, object_rows = [], []
    ann_dir = root_dir / 'Annotations'

    for image_id in image_ids:
        xml_path = ann_dir / f'{image_id}.xml'
        if not xml_path.exists():
            image_rows.append({'image_id': image_id, 'split': split_name, 'width': np.nan, 'height': np.nan})
            continue

        image_meta, rows = parse_annotation(xml_path, image_id, split_name)
        image_rows.append(image_meta)
        object_rows.extend(rows)

    images_df = pd.DataFrame(image_rows).drop_duplicates(subset=['image_id', 'split'])
    objects_df = pd.DataFrame(object_rows)
    return images_df, objects_df


train_ids = load_split_ids('train')
val_ids = load_split_ids('val')
test_ids = load_split_ids('test')

if not train_ids or not val_ids or not test_ids:
    raise RuntimeError('Missing one or more split files. Check artifacts/group_split or ImageSets/Main.')

train_images, train_objects = build_split_tables(TRAINVAL_ROOT, 'train', train_ids)
val_images, val_objects = build_split_tables(TRAINVAL_ROOT, 'val', val_ids)
test_images, test_objects = build_split_tables(TEST_ROOT, 'test', test_ids)

train_model_df = train_objects[(train_objects['xmax'] >= train_objects['xmin']) & (train_objects['ymax'] >= train_objects['ymin'])].copy()
val_model_df = val_objects[(val_objects['xmax'] >= val_objects['xmin']) & (val_objects['ymax'] >= val_objects['ymin'])].copy()

train_model_df_ignoring_difficult = train_model_df[train_model_df['difficult'] == 0].reset_index(drop=True)
val_model_df_ignoring_difficult = val_model_df[val_model_df['difficult'] == 0].reset_index(drop=True)

print('Split counts:', len(train_ids), len(val_ids), len(test_ids))
print('train-val overlap:', len(set(train_ids) & set(val_ids)))
print('train-test overlap:', len(set(train_ids) & set(test_ids)))
print('val-test overlap:', len(set(val_ids) & set(test_ids)))

display(train_model_df_ignoring_difficult.head())

In [ ]:
class VOCDetectionDataset(Dataset):
    def __init__(self, root: Path, image_ids: list[str], objects_df: pd.DataFrame, transforms=None):
        self.root = root
        self.image_ids = list(image_ids)
        self.transforms = transforms
        self.objects_by_image = {}

        if not objects_df.empty:
            for image_id, group in objects_df.groupby('image_id'):
                self.objects_by_image[image_id] = group.copy()

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_path = self.root / 'JPEGImages' / f'{image_id}.jpg'
        image = Image.open(image_path).convert('RGB')

        rows = self.objects_by_image.get(image_id)
        if rows is None or rows.empty:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
        else:
            boxes = torch.tensor(rows[['xmin', 'ymin', 'xmax', 'ymax']].to_numpy(), dtype=torch.float32)
            labels = torch.tensor(rows['class_id'].to_numpy(), dtype=torch.int64)
            area = torch.tensor(rows['box_area'].to_numpy(), dtype=torch.float32)

        target = {
            'boxes': boxes,
            'labels': labels,
            'area': area,
            'iscrowd': torch.zeros((boxes.shape[0],), dtype=torch.int64),
            'image_id': image_id,
        }

        if self.transforms is not None:
            image, target = self.transforms(image, target)

        return image, target


def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)


train_ds = VOCDetectionDataset(TRAINVAL_ROOT, train_ids, train_model_df_ignoring_difficult)
val_ds = VOCDetectionDataset(TRAINVAL_ROOT, val_ids, val_model_df_ignoring_difficult)
test_ds = VOCDetectionDataset(TEST_ROOT, test_ids, test_objects)

print('Dataset sizes:', len(train_ds), len(val_ds), len(test_ds))

In [ ]:
def compute_iou_xyxy(box_a: np.ndarray, box_b: np.ndarray) -> float:
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b

    inter_x1 = max(xa1, xb1)
    inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2)
    inter_y2 = min(ya2, yb2)

    inter_w = max(0.0, inter_x2 - inter_x1 + 1)
    inter_h = max(0.0, inter_y2 - inter_y1 + 1)
    inter_area = inter_w * inter_h

    area_a = max(0.0, xa2 - xa1 + 1) * max(0.0, ya2 - ya1 + 1)
    area_b = max(0.0, xb2 - xb1 + 1) * max(0.0, yb2 - yb1 + 1)

    union = area_a + area_b - inter_area
    return 0.0 if union <= 0 else inter_area / union


def ap_from_pr(precisions: np.ndarray, recalls: np.ndarray) -> float:
    mrec = np.concatenate(([0.0], recalls, [1.0]))
    mpre = np.concatenate(([0.0], precisions, [0.0]))

    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])

    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))


def evaluate_map50(pred_df: pd.DataFrame, gt_df: pd.DataFrame, classes: list[str], iou_thresh: float = 0.5) -> dict:
    if pred_df.empty:
        return {'map50': 0.0, 'per_class_ap': {cls: 0.0 for cls in classes}}

    per_class_ap = {}
    for cls in classes:
        gt_cls = gt_df[gt_df['cls'] == cls]
        pred_cls = pred_df[pred_df['cls'] == cls].sort_values('score', ascending=False)

        gt_by_image = {}
        for image_id, group in gt_cls.groupby('image_id'):
            gt_by_image[image_id] = {
                'boxes': group[['xmin', 'ymin', 'xmax', 'ymax']].to_numpy(dtype=float),
                'matched': np.zeros(len(group), dtype=bool),
            }

        tps = np.zeros(len(pred_cls), dtype=float)
        fps = np.zeros(len(pred_cls), dtype=float)

        for i, row in enumerate(pred_cls.itertuples(index=False)):
            image_state = gt_by_image.get(row.image_id)
            if image_state is None or len(image_state['boxes']) == 0:
                fps[i] = 1.0
                continue

            pred_box = np.array([row.xmin, row.ymin, row.xmax, row.ymax], dtype=float)
            ious = np.array([compute_iou_xyxy(pred_box, gt_box) for gt_box in image_state['boxes']])
            best_idx = int(np.argmax(ious))
            best_iou = ious[best_idx]

            if best_iou >= iou_thresh and not image_state['matched'][best_idx]:
                tps[i] = 1.0
                image_state['matched'][best_idx] = True
            else:
                fps[i] = 1.0

        cum_tp = np.cumsum(tps)
        cum_fp = np.cumsum(fps)
        recalls = cum_tp / max(1, len(gt_cls))
        precisions = cum_tp / np.maximum(cum_tp + cum_fp, 1e-12)
        per_class_ap[cls] = ap_from_pr(precisions, recalls) if len(pred_cls) > 0 else 0.0

    map50 = float(np.mean(list(per_class_ap.values()))) if per_class_ap else 0.0
    return {'map50': map50, 'per_class_ap': per_class_ap}


def ensure_prediction_schema(df: pd.DataFrame, source_model: str, split_name: str) -> pd.DataFrame:
    out = df.copy()
    out['source_model'] = source_model
    out['split'] = split_name
    for col in PREDICTION_COLUMNS:
        if col not in out.columns:
            out[col] = np.nan
    out = out[PREDICTION_COLUMNS]
    out['class_id'] = out['class_id'].astype('Int64')
    return out

## DETR Template

Recommended package: transformers.

Expected outputs:
- artifacts/predictions/detr_val_predictions.csv
- artifacts/predictions/detr_test_predictions.csv

In [ ]:
# Optional installs (run once):
# %pip install transformers torchvision accelerate

from transformers import DetrForObjectDetection, DetrImageProcessor

processor = DetrImageProcessor.from_pretrained('facebook/detr-resnet-50')
model = DetrForObjectDetection.from_pretrained(
    'facebook/detr-resnet-50',
    num_labels=len(VOC_CLASSES),
    ignore_mismatched_sizes=True,
)
model.to(DEVICE)

# Create your train/val dataloaders and implement a full fine-tuning loop.
# Keep batch-level outputs convertible to the shared prediction schema below.

In [ ]:
# Template stub: replace with your actual DETR inference outputs.
# Required columns before schema function:
# image_id, class_id, cls, xmin, ymin, xmax, ymax, score

val_pred_raw = pd.DataFrame(columns=['image_id', 'class_id', 'cls', 'xmin', 'ymin', 'xmax', 'ymax', 'score'])
test_pred_raw = pd.DataFrame(columns=['image_id', 'class_id', 'cls', 'xmin', 'ymin', 'xmax', 'ymax', 'score'])

val_pred_df = ensure_prediction_schema(val_pred_raw, source_model='detr', split_name='val')
test_pred_df = ensure_prediction_schema(test_pred_raw, source_model='detr', split_name='test')

val_path = PREDICTIONS_DIR / 'detr_val_predictions.csv'
test_path = PREDICTIONS_DIR / 'detr_test_predictions.csv'
val_pred_df.to_csv(val_path, index=False)
test_pred_df.to_csv(test_path, index=False)

metrics = evaluate_map50(val_pred_df, val_model_df_ignoring_difficult, VOC_CLASSES)
print('DETR val mAP@0.5:', round(metrics['map50'], 4))
print('Saved:', val_path)
print('Saved:', test_path)